# IMPORTS


In [ ]:
# %%
%pip install -q ultralytics seaborn scikit-learn orjson
import os
import orjson
import shutil
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from collections import Counter, defaultdict
from multiprocessing import Pool, cpu_count
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# Data Paths


In [ ]:
# --- Paths ---
DATA_ROOT = "./backup"
TRAIN_IMG = f"{DATA_ROOT}/train/image"
TRAIN_ANN = f"{DATA_ROOT}/train/annos"
VAL_IMG = f"{DATA_ROOT}/validation/image"
VAL_ANN = f"{DATA_ROOT}/validation/annos"
WORK_DIR = "./yolo_dataset"

os.makedirs(WORK_DIR, exist_ok=True)
print("CUDA:", torch.cuda.is_available())


# Data preprocessing


## Parsing functions


In [ ]:
# --- Parsing Function ---
def parse_file(args):
    file, image_dir, annot_dir = args
    with open(os.path.join(annot_dir, file), "rb") as f:
        ann = orjson.loads(f.read())

    image_name = file.replace(".json", ".jpg")
    local_data = []

    for key in ann:
        if "item" not in key:
            continue
        item = ann[key]
        category = item["category_id"]
        segmentation = item.get("segmentation", [])

        local_data.append({
            "image": image_name,
            "category": category,
            "segmentation": segmentation,
            "img_dir": image_dir
        })

    return local_data

def parse_annotations(image_dir, annot_dir):
    files = os.listdir(annot_dir)
    data = []
    
    with Pool(cpu_count()) as p:
        for d in tqdm(p.imap(parse_file, [(f, image_dir, annot_dir) for f in files]), total=len(files), desc=f"Parsing {os.path.basename(annot_dir)}"):
            data.extend(d)

    return data

## Parse all the data


In [ ]:
# 1. Parse all data
train_data_full = parse_annotations(TRAIN_IMG, TRAIN_ANN)
val_data = parse_annotations(VAL_IMG, VAL_ANN)

category_map = {
    1: 0,
    8: 1,
    7: 2,
    2: 3, 
    9: 4
}
print("Top 5 Categories:", category_map)

## Split the train data


In [ ]:
train_top = [d for d in train_data_full if d['category'] in category_map]
print(len(train_top))

unique_train_img = list(set([d['image'] for d in train_top]))

train_imgs, test_imgs = train_test_split(unique_train_img, test_size=0.1, random_state=42)

print(len(train_imgs), len(test_imgs))

In [ ]:
train_set = set(train_imgs)

train_split = [d for d in train_top if d['image'] in train_set]
val_split = [d for d in val_data if d['category'] in category_map]

print(len(train_split), len(val_split))

## Store the test data


In [ ]:
# 3. Handle the Test Split (Simply Store It)
test_img_dir = f"{WORK_DIR}/test/images"
test_ann_dir = f"{WORK_DIR}/test/annos"
os.makedirs(test_img_dir, exist_ok=True)
os.makedirs(test_ann_dir, exist_ok=True)

for img_name in tqdm(test_imgs, desc="Saving Test Split"):
    src_img = os.path.join(TRAIN_IMG, img_name)
    src_ann = os.path.join(TRAIN_ANN, img_name.replace('.jpg', '.json'))
    
    if os.path.exists(src_img) and os.path.exists(src_ann):
        shutil.copy(src_img, os.path.join(test_img_dir, img_name))
        shutil.copy(src_ann, os.path.join(test_ann_dir, img_name.replace('.jpg', '.json')))

## Preprocess for YOLO model


In [ ]:
# 4. YOLO Preprocessing Functions
def group_by_image(dataset):
    grouped = defaultdict(list)
    for d in dataset:
        grouped[d["image"]].append(d)
    return grouped

def get_yolo_polygons(polygons, w, h):
    """
    Fixes the 0.0 mask mAP issue by drawing complex/disjoint polygons 
    onto a mask and extracting clean external contours.
    """
    mask = np.zeros((h, w), dtype=np.uint8)
    
    for poly in polygons:
        if len(poly) < 6: continue
        pts = np.array(poly, np.int32).reshape((-1, 2))
        cv2.fillPoly(mask, [pts], 255)
        
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    norm_polys = []
    for contour in contours:
        contour = contour.flatten()
        if len(contour) < 6: continue
        
        norm_poly = []
        for i in range(0, len(contour), 2):
            x = max(0.0, min(1.0, contour[i] / w))
            y = max(0.0, min(1.0, contour[i+1] / h))
            norm_poly.extend([x, y])
        norm_polys.append(norm_poly)
        
    return norm_polys

def process_one(args):
    img, records, split_name, cat_map, work_dir = args
    src = os.path.join(records[0]["img_dir"], img)

    if not os.path.exists(src): return

    dst_img_dir = f"{work_dir}/images/{split_name}"
    dst_lbl_dir = f"{work_dir}/labels/{split_name}"
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    dst = os.path.join(dst_img_dir, img)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

    with Image.open(src) as im:
        w, h = im.size

    label_path = os.path.join(dst_lbl_dir, img.replace('.jpg', '.txt'))
    lines = []

    for d in records:
        if d["category"] not in cat_map: continue
        cid = cat_map[d["category"]]
        
        # Get clean contours
        clean_polys = get_yolo_polygons(d["segmentation"], w, h)
        
        # Write each contour as a label line
        for poly in clean_polys:
            lines.append(f"{cid} " + " ".join(map(str, poly)))

    if lines:
        with open(label_path, "w") as f:
            f.write("\n".join(lines))

def process_split(dataset, split_name):
    grouped = group_by_image(dataset)
    items = [(img, records, split_name, category_map, WORK_DIR) for img, records in grouped.items()]
    with Pool(cpu_count()) as p:
        list(tqdm(p.imap(process_one, items), total=len(items), desc=f"Writing YOLO {split_name} data"))

## Process the train and val split


In [ ]:
# 5. Process Train and Val for YOLO
process_split(train_split, "train")


In [ ]:
print(len(val_split))
print(len([d['image'] for d in val_split]))
process_split(val_split, "val")


## Make the YAML config file


In [ ]:
# 6. Create YAML Configuration
yaml_text=f"""
path: {WORK_DIR}
train: images/train
val: images/val

names:
    0: short sleeve top
    1: trousers
    2: shorts
    3: long sleeve top 
    4: skirt
"""

with open(f"{WORK_DIR}/data.yaml","w") as f:
    f.write(yaml_text)

print("Data processing complete. Ready for YOLO training.")

## Train Args


In [ ]:
# %%
train_args = dict(
    data=f"{WORK_DIR}/data.yaml",
    epochs=15,
    patience=3,
    imgsz=512,
    batch=96,
    device="0,1",
    amp=True,
    workers=4
)

In [ ]:
def extract(m):
    return {
        "box_mAP50": m.box.map50,
        "mask_mAP50": m.seg.map50,
        "precision": m.box.mp,
        "recall": m.box.mr
    }

## Train the model using tranfer learning


In [ ]:
model_transfer = YOLO("yolov8n-seg.pt")
model_transfer.train(**train_args, freeze=15, lr0=0.001)

In [ ]:
m2 = model_transfer.val()
pd.DataFrame([
    {"Model":"Transfer", **extract(m2)},
])

## Train the model from scratch


In [ ]:
model_scratch = YOLO("yolov8n-seg.yaml")
model_scratch.train(**train_args)

In [ ]:
m1 = model_scratch.val()
pd.DataFrame([
    {"Model":"Scratch", **extract(m1)},
])